# Extra · The jet stream

The **jet streams** are bands of strong westerly wind around the globe. The **eddy-driven jet** of the
mid-latitudes steers the storms that bring rain to Europe. How to find it, and where it is, is a research
topic of its own ([Adam et al., 2018](https://gmd.copernicus.org/articles/11/4339/2018/);
[Breul et al., 2025](https://www.nature.com/articles/s43017-025-00749-9)). In this notebook you

1. map the **zonal wind** (from west to east) at 700 hPa (about 3 km high) in winter and summer,
2. compute its **zonal mean**: the mean around each latitude,
3. find the **latitude and strength** of the jet in each hemisphere and each month, and
4. see how they changed since 1950.

Data: ERA5 zonal wind `u` at 700 hPa, monthly 1950–2023, at zoom 6 (enough for a jet thousands of km long).

## 1. The data

`u` has a vertical dimension, `level`, in hPa. We take 700 hPa and load all months into memory (about 175 MB;
it can take a minute).

In [ ]:
import cartopy.crs as ccrs
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
u = cat.ERA5(zoom=6).to_dask()["u"].sel(level=700, time=slice("1950", "2023")).compute()
u.attrs

## 2. Maps: winter and summer

`u.time.dt.month.isin([12, 1, 2])` keeps the months of December, January and February (DJF, winter in the
north); JJA is June, July, August. Red: wind from the west; blue: from the east.

In [ ]:
seasons = {"DJF": [12, 1, 2], "JJA": [6, 7, 8]}

fig, axes = plt.subplots(
    1, 2, figsize=(16, 4), subplot_kw={"projection": ccrs.Robinson()}
)
for ax, (season, months) in zip(axes, seasons.items()):
    mean = u.sel(time=u.time.dt.month.isin(months)).mean("time")
    ax.set_global()
    ax.coastlines(linewidth=0.4)
    image = egh.healpix_show(mean, ax=ax, cmap="RdBu_r", vmin=-20, vmax=20)
    ax.set_title(f"Zonal wind at 700 hPa, {season}, 1950–2023")
fig.colorbar(image, ax=axes, shrink=0.7, label="u / m per s")

fig.savefig("/app/pages/jet_stream/u_maps.png", dpi=100, bbox_inches="tight")
plt.show()

## 3. The zonal mean

HEALPix cells lie on rings of equal latitude: `.groupby("lat").mean()` averages each ring. The result has one
value per latitude and month.

In [ ]:
zonal = u.groupby("lat").mean()

fig, ax = plt.subplots(figsize=(6, 5))
for season, months in seasons.items():
    zonal.sel(time=zonal.time.dt.month.isin(months)).mean("time").plot(
        ax=ax, y="lat", label=season
    )
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("zonal mean u / m per s")
ax.set_ylabel("latitude / °N")
ax.set_title("Zonal mean wind at 700 hPa, 1950–2023")
ax.legend()

fig.savefig("/app/pages/jet_stream/zonal_mean.png", dpi=100, bbox_inches="tight")
plt.show()

Two maxima: the eddy-driven jets of each hemisphere, between about 30° and 60°. The southern one is
stronger, and both move towards the pole in their summer.

## 4. Latitude and strength of the jet

The **strength** is the largest wind of the hemisphere (between 20° and 70°). The **latitude** is a weighted
mean of the latitudes, with the wind to the power $n = 8$ as weight: the strongest winds count almost alone
(Adam et al., 2018). `clip(min=0)` sets the winds from the east to 0, so they do not count.

In [ ]:
def jet(zonal, south, north, n=8):
    band = zonal.sel(lat=slice(south, north))
    weights = band.clip(min=0) ** n
    latitude = (weights * band.lat).sum("lat") / weights.sum("lat")
    strength = band.max("lat")
    return latitude, strength


jets = {"north": jet(zonal, 20, 70), "south": jet(zonal, -70, -20)}

Yearly means of every month's jet, with a trend for each:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)
for column, (hemisphere, (latitude, strength)) in enumerate(jets.items()):
    for row, (data, label) in enumerate(
        [(latitude, "latitude / °N"), (strength, "strength / m per s")]
    ):
        yearly = data.resample(time="YS").mean()
        years = yearly.time.dt.year.values
        slope, intercept = np.polyfit(years, yearly.values, 1)
        ax = axes[row, column]
        ax.plot(years, yearly, marker=".")
        ax.plot(
            years,
            slope * years + intercept,
            color="black",
            label=f"{10 * slope:+.2f} per decade",
        )
        ax.set_ylabel(label)
        ax.set_title(f"Jet of the {hemisphere}")
        ax.legend()

fig.savefig("/app/pages/jet_stream/jet_timeseries.png", dpi=100, bbox_inches="tight")
plt.show()

for hemisphere, (latitude, strength) in jets.items():
    change_lat = float(
        latitude.sel(time=slice("1981", "2023")).mean()
        - latitude.sel(time=slice("1950", "1980")).mean()
    )
    change_u = float(
        strength.sel(time=slice("1981", "2023")).mean()
        - strength.sel(time=slice("1950", "1980")).mean()
    )
    print(
        f"{hemisphere}: 1981-2023 minus 1950-1980: latitude {change_lat:+.2f}°, strength {change_u:+.2f} m/s"
    )

## 5. Exercise

Higher up, at **250 hPa** (about 10 km), is the **subtropical jet**, much stronger. Load `u` at `level=250`,
draw its zonal mean in DJF and JJA as in section 3, and find its latitude and strength. Is it closer to the
equator than the eddy-driven jet? Save your plot as `/app/pages/jet_stream/jet_exercise.png`.

In [ ]:
# the jet at 250 hPa

## Your page

Open `pages/jet_stream/my_page.rst` in JupyterLab and describe what you found. Look at it on
http://localhost:8000 → *Extras*.